# NLTK — The Swiss Army Knife of Classic NLP

---

## What Is NLTK?

NLTK (Natural Language Toolkit) is one of the oldest and most comprehensive Python libraries for **Natural Language Processing (NLP)**. Created in 2001 at the University of Pennsylvania, it was designed primarily for teaching and research in computational linguistics.

NLTK gives you:
- Tools to **process text**: tokenize, tag parts of speech, parse grammar
- Access to **50+ corpora** (text datasets like news articles, books, movie reviews)
- **Classical NLP algorithms**: stemming, lemmatization, named entity recognition
- **Statistical classifiers**: Naive Bayes, Maximum Entropy for text tasks
- Building blocks to understand how NLP works from the ground up

### Real-World Analogy

Imagine you need to analyze a library of books. Before you can understand meaning, you need to:
1. Split text into sentences and words (tokenization)
2. Identify what each word IS: noun, verb, adjective (part-of-speech tagging)
3. Find the base form of words: "running" → "run" (lemmatization)
4. Identify people, places, organizations in text (named entity recognition)

NLTK is the **librarian's toolkit** for all these foundational text processing tasks. Modern libraries like spaCy and HuggingFace Transformers are faster and more accurate for production, but NLTK is still the best starting point for **understanding** NLP concepts.

---

## NLTK vs spaCy vs Transformers

| Feature | NLTK | spaCy | HuggingFace Transformers |
|---|---|---|---|
| Speed | Slow | Fast | Slowest (GPU helps) |
| Accuracy | Moderate | High | Highest |
| Use case | Learning, research | Production NLP | State-of-the-art AI |
| Ease of use | Moderate | Very easy | Easy (high-level) |
| Corpora access | 50+ built-in | Few | HuggingFace Hub (millions) |
| Best for | Teaching, classic NLP | Production pipelines | BERT/GPT/LLM tasks |

---

## Prerequisites

- Python basics
- What a string is; basic string operations
- No prior NLP experience needed!

---

## Table of Contents

1. Installation & Setup
2. Tokenization — Splitting Text into Pieces
3. Stopwords — Filtering Noise
4. Stemming vs Lemmatization — Finding Root Words
5. Part-of-Speech (POS) Tagging
6. Named Entity Recognition (NER)
7. Frequency Analysis & Collocations
8. Sentiment Analysis with NLTK
9. Text Classification Pipeline
10. NLTK Corpora — Built-in Datasets
11. Mini Project — News Article Analyzer
12. Common Pitfalls
13. Interview Q&A
14. Resources
15. Summary & What's Next

---

**Official Docs:** https://www.nltk.org/  
**Book (free online):** https://www.nltk.org/book/  
**GitHub:** https://github.com/nltk/nltk  
**YouTube — NLP with NLTK (Sentdex):** https://www.youtube.com/watch?v=FLZvOKSCkxY  

## 1. Installation & Setup

```bash
pip install nltk
```

NLTK separates the library code from its **data** (corpora, trained models). You download data separately using `nltk.download()`.

In [ ]:
import nltk
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import string
import warnings
warnings.filterwarnings('ignore')

# Download required NLTK data packages
packages = [
    'punkt',           # tokenizer models
    'punkt_tab',       # newer punkt format
    'stopwords',       # stopword lists
    'wordnet',         # WordNet lexical database (for lemmatization)
    'averaged_perceptron_tagger',  # POS tagger
    'averaged_perceptron_tagger_eng',
    'maxent_ne_chunker',  # Named entity recognizer
    'maxent_ne_chunker_tab',
    'words',           # English word list
    'vader_lexicon',   # Sentiment lexicon
    'movie_reviews',   # Corpus for sentiment
    'brown',           # Brown corpus
    'gutenberg',       # Project Gutenberg books
]

for pkg in packages:
    try:
        nltk.download(pkg, quiet=True)
    except:
        pass

print(f"NLTK version: {nltk.__version__}")
print("Required packages downloaded!")

## 2. Tokenization — Splitting Text into Pieces

**Tokenization** is the first step in almost every NLP pipeline. It means splitting raw text into meaningful units:
- **Word tokenization**: split into individual words and punctuation
- **Sentence tokenization**: split into individual sentences

It sounds simple, but edge cases are tricky: `"Dr. Smith went to Washington D.C. yesterday."` — how many sentences? What about `"I'm"`? Contractions? Emojis?

In [ ]:
from nltk.tokenize import word_tokenize, sent_tokenize, TweetTokenizer, MWETokenizer

text = """
Dr. Smith went to Washington D.C. for the AI conference yesterday.
She presented her findings on large language models (LLMs).
"It's amazing!", she said. The audience applauded.
The model achieved 98.5% accuracy on the test set.
""".strip()

# Sentence tokenization
sentences = sent_tokenize(text)
print("=== Sentence Tokenization ===")
for i, s in enumerate(sentences):
    print(f"  [{i}]: {s}")

# Word tokenization
words = word_tokenize(text)
print(f"\n=== Word Tokenization ===")
print(f"Total tokens: {len(words)}")
print(f"First 20: {words[:20]}")

# Note: punctuation IS kept as separate tokens
print(f"\nTokens that are punctuation: {[w for w in words if w in string.punctuation]}")

In [ ]:
# Tweet Tokenizer — handles social media text better
tweet = "I can't believe @nltk_team released v3.8!! #NLP is amazing 😍 http://nltk.org"

# Standard tokenizer (treats @ and # as punctuation)
print("Standard:", word_tokenize(tweet))

# Tweet tokenizer (preserves hashtags, handles contractions, emojis)
tweet_tok = TweetTokenizer(strip_handles=False, reduce_len=True)
print("Tweet:    ", tweet_tok.tokenize(tweet))

## 3. Stopwords — Filtering Noise

**Stopwords** are extremely common words that carry little meaning: *the, a, is, in, at, which, on, and, ...*. Removing them reduces noise and speeds up downstream processing.

NLTK provides stopword lists for 23 languages.

In [ ]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
print(f"Total English stopwords: {len(stop_words)}")
print(f"Sample: {sorted(list(stop_words))[:20]}")

# Filter stopwords from text
sample = "The quick brown fox jumps over the lazy dog and then runs away quickly"
tokens = word_tokenize(sample.lower())

filtered = [w for w in tokens
            if w not in stop_words and w not in string.punctuation]

print(f"\nOriginal: {tokens}")
print(f"Filtered: {filtered}")
print(f"Reduction: {len(tokens)} → {len(filtered)} tokens ({100*(1-len(filtered)/len(tokens)):.0f}% removed)")

# ---- Pipeline function ----
def preprocess(text, language='english'):
    """Clean text: lowercase → tokenize → remove stopwords/punctuation."""
    stop = set(stopwords.words(language))
    tokens = word_tokenize(text.lower())
    return [w for w in tokens if w.isalpha() and w not in stop]

result = preprocess(sample)
print(f"\nPreprocessed: {result}")

## 4. Stemming vs Lemmatization — Finding Root Words

Both reduce words to their base form so that `"running"`, `"runs"`, `"ran"` are treated as the same concept.

| | Stemming | Lemmatization |
|---|---|---|
| Method | Chops off suffixes by rule | Uses a dictionary (WordNet) |
| Output | May not be a real word: `"studi"` | Always a real word: `"study"` |
| Accuracy | Lower (aggressive) | Higher |
| Speed | Faster | Slower |
| Context | Ignores word meaning | Uses POS tag for accuracy |

**Rule of thumb:** Use lemmatization for quality NLP, stemming when speed matters.

In [ ]:
from nltk.stem import PorterStemmer, SnowballStemmer, LancasterStemmer
from nltk.stem import WordNetLemmatizer

test_words = ['running', 'runs', 'ran', 'easily', 'fairly', 'studies',
              'studying', 'studied', 'better', 'caring', 'believes']

# Three stemmers with different aggressiveness
porter   = PorterStemmer()
snowball = SnowballStemmer('english')
lancaster = LancasterStemmer()
lemmatizer = WordNetLemmatizer()

print(f"{'Word':<12} {'Porter':<12} {'Snowball':<12} {'Lancaster':<12} {'Lemma(v)':<12} {'Lemma(n)':<12}")
print("-" * 72)
for word in test_words:
    p = porter.stem(word)
    s = snowball.stem(word)
    l = lancaster.stem(word)
    lm_v = lemmatizer.lemmatize(word, pos='v')  # treat as verb
    lm_n = lemmatizer.lemmatize(word, pos='n')  # treat as noun
    print(f"{word:<12} {p:<12} {s:<12} {l:<12} {lm_v:<12} {lm_n:<12}")

## 5. Part-of-Speech (POS) Tagging

POS tagging labels each word with its grammatical role: noun (NN), verb (VB), adjective (JJ), adverb (RB), etc.

Why does this matter?
- Disambiguation: "*bank*" (noun = financial institution? / verb = to bank on?)
- Better lemmatization: knowing "better" is an adjective → lemma is "good"
- Feature engineering for ML models
- Named entity recognition pipelines

In [ ]:
from nltk import pos_tag

sentence = "The quick brown fox jumped over the lazy dog near the river bank"
tokens = word_tokenize(sentence)
tags = pos_tag(tokens)

print("POS Tags:")
for word, tag in tags:
    print(f"  {word:<12} → {tag}")

# Common POS tags:
tag_meanings = {
    'NN': 'Noun (singular)',    'NNS': 'Noun (plural)',
    'NNP': 'Proper noun',       'VB': 'Verb (base)',
    'VBD': 'Verb (past)',       'VBG': 'Verb (gerund)',
    'VBN': 'Verb (past part.)', 'VBZ': 'Verb (3rd sing.)',
    'JJ': 'Adjective',         'JJR': 'Adj (comparative)',
    'JJS': 'Adj (superlative)', 'RB': 'Adverb',
    'DT': 'Determiner',        'IN': 'Preposition/Conjunction',
    'PRP': 'Pronoun',          'CC': 'Coordinating conjunction'
}

print("\nCommon POS Tag Reference:")
for tag, meaning in tag_meanings.items():
    print(f"  {tag:<6} → {meaning}")

In [ ]:
# POS-aware lemmatization (much better than naive lemmatization)

def get_wordnet_pos(treebank_tag):
    """Convert NLTK POS tags to WordNet format for lemmatizer."""
    from nltk.corpus import wordnet
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN  # default

def pos_lemmatize(text):
    """Lemmatize using POS context for each word."""
    lemmatizer = WordNetLemmatizer()
    tokens = word_tokenize(text)
    tags   = pos_tag(tokens)
    return [lemmatizer.lemmatize(w, pos=get_wordnet_pos(t)) for w, t in tags]

test_sentence = "The geese were running better than they had ever run before."
naive_lemma = [WordNetLemmatizer().lemmatize(w) for w in word_tokenize(test_sentence)]
pos_aware   = pos_lemmatize(test_sentence)

print(f"Original:   {word_tokenize(test_sentence)}")
print(f"Naive:      {naive_lemma}")
print(f"POS-aware:  {pos_aware}")
print("\nNotice: POS-aware correctly lemmatizes 'ran'→'run', 'better'→'good'")

## 6. Named Entity Recognition (NER)

NER identifies and classifies **named entities** in text:
- **PERSON**: `Albert Einstein`, `Dr. Smith`
- **ORGANIZATION**: `Google`, `United Nations`
- **GPE** (Geo-political entity): `New York`, `France`
- **DATE**: `January 2024`, `yesterday`
- **MONEY**: `$50 million`

NER is used in: information extraction, knowledge graph building, document indexing, compliance screening.

In [ ]:
from nltk import ne_chunk
from nltk.tree import Tree

ner_text = """
Elon Musk, the CEO of Tesla and SpaceX, announced yesterday that his company
will invest $5 billion in Austin, Texas. The deal was confirmed by the U.S.
Department of Energy and Goldman Sachs.
"""

tokens = word_tokenize(ner_text.strip())
pos_tags = pos_tag(tokens)
chunks = ne_chunk(pos_tags, binary=False)  # binary=False → gives entity types

# Extract named entities
entities = []
for subtree in chunks:
    if isinstance(subtree, Tree):
        entity_text = ' '.join([word for word, tag in subtree.leaves()])
        entity_type = subtree.label()
        entities.append((entity_text, entity_type))

print("Named Entities Found:")
print(f"{'Entity':<25} {'Type':<15}")
print("-" * 40)
for entity, etype in entities:
    print(f"{entity:<25} {etype:<15}")

print("\nNote: NLTK's NER is basic. For production, use spaCy or HuggingFace!")

## 7. Frequency Analysis & Collocations

**Frequency distribution**: how often does each word appear?  
**Collocations**: word pairs that appear together more often than expected by chance (e.g., "New York", "United States", "machine learning")

In [ ]:
from nltk import FreqDist, bigrams, BigramCollocationFinder
from nltk.metrics import BigramAssocMeasures
from nltk.corpus import gutenberg

# Use a classic text (Moby Dick is built into NLTK via Gutenberg corpus)
moby_words = gutenberg.words('melville-moby_dick.txt')
print(f"Moby Dick: {len(moby_words):,} words")

# Clean: lowercase, alphabetic only
stop = set(stopwords.words('english'))
clean = [w.lower() for w in moby_words if w.isalpha() and w.lower() not in stop]

# Frequency distribution
fdist = FreqDist(clean)
print(f"\nTop 15 most common meaningful words:")
print(fdist.most_common(15))

# Plot
fig, ax = plt.subplots(figsize=(12, 4))
top_words, top_counts = zip(*fdist.most_common(20))
ax.bar(range(20), top_counts, color='steelblue')
ax.set_xticks(range(20))
ax.set_xticklabels(top_words, rotation=45, ha='right')
ax.set_title('Top 20 Most Frequent Words in Moby Dick (stopwords removed)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Bigram collocations — pairs that appear together often
finder = BigramCollocationFinder.from_words(clean)

# Apply frequency filter (ignore pairs that appear fewer than 5 times)
finder.apply_freq_filter(5)

# Score using Pointwise Mutual Information (PMI) — measures how much
# more often the pair co-occurs compared to random chance
bigram_measures = BigramAssocMeasures()
top_collocations = finder.nbest(bigram_measures.pmi, 15)

print("Top 15 Bigram Collocations in Moby Dick (by PMI):")
for i, (w1, w2) in enumerate(top_collocations):
    print(f"  {i+1:2d}. {w1} {w2}")

## 8. Sentiment Analysis with VADER

VADER (Valence Aware Dictionary and sEntiment Reasoner) is a rule-based sentiment analyzer built into NLTK, specially tuned for **social media text**. It understands:
- Capitalization: `"AMAZING"` is stronger than `"amazing"`
- Punctuation: `"amazing!!!"` is stronger than `"amazing"`
- Negations: `"not good"` is negative
- Slang and emojis: `"😍"` is positive

Output: `neg`, `neu`, `pos` scores (sum to 1.0) + compound score (-1.0 to 1.0)

In [ ]:
from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

test_sentences = [
    "This product is absolutely amazing! Best purchase I've ever made!",
    "The movie was terrible. I want my money back.",
    "The package arrived on time.",          # neutral
    "Not bad at all!",                        # tricky: negation
    "I LOVE this SO MUCH!!!",                # caps + punctuation
    "It's okay but could be better.",        # mixed
    "Absolutely disgusting service 😡",       # emoji
    "Best day ever 😍",                       # positive emoji
]

print(f"{'Text':<45} {'Compound':>10} {'Label':>10}")
print("-" * 70)
for text in test_sentences:
    scores   = sia.polarity_scores(text)
    compound = scores['compound']
    label    = 'Positive' if compound >= 0.05 else 'Negative' if compound <= -0.05 else 'Neutral'
    display  = text[:43] + '...' if len(text) > 45 else text
    print(f"{display:<45} {compound:>10.3f} {label:>10}")

## 9. Text Classification with NLTK's Naive Bayes

NLTK includes simple classifiers. Here we build a **bag-of-words** text classifier using Naive Bayes — a probabilistic model that assumes each word is independent.

In [ ]:
from nltk.corpus import movie_reviews
from nltk import NaiveBayesClassifier, classify
import random

# Load NLTK's movie reviews corpus (2000 positive/negative reviews)
documents = [(list(movie_reviews.words(fileid)), category)
             for category in movie_reviews.categories()
             for fileid in movie_reviews.fileids(category)]

random.seed(42)
random.shuffle(documents)
print(f"Total reviews: {len(documents)}")
print(f"Categories: {movie_reviews.categories()}")

# Build word features: top 2000 most common words
all_words = FreqDist(w.lower() for w in movie_reviews.words())
word_features = [w for w, _ in all_words.most_common(2000)]

def document_features(doc):
    """Bag-of-words feature: which top words appear in this document?"""
    doc_set = set(w.lower() for w in doc)
    return {f'contains({w})': (w in doc_set) for w in word_features}

# Create feature sets
feature_sets = [(document_features(doc), label) for doc, label in documents]

# Train/test split (80/20)
train_set = feature_sets[:1600]
test_set  = feature_sets[1600:]

# Train Naive Bayes classifier
nb_classifier = NaiveBayesClassifier.train(train_set)

accuracy = classify.accuracy(nb_classifier, test_set)
print(f"\nNaive Bayes Accuracy: {accuracy:.4f} ({accuracy*100:.1f}%)")

print("\nMost informative features:")
nb_classifier.show_most_informative_features(10)

## 10. NLTK Corpora — Built-in Datasets

In [ ]:
# NLTK comes with 50+ built-in corpora

print("=== Gutenberg Corpus (classic literature) ===")
from nltk.corpus import gutenberg
print("Books:", gutenberg.fileids())
print(f"Hamlet words: {len(gutenberg.words('shakespeare-hamlet.txt')):,}")

print("\n=== Brown Corpus (balanced general English) ===")
from nltk.corpus import brown
print("Categories:", brown.categories())
print(f"Total words: {len(brown.words()):,}")

# POS tag distribution in news articles
news_tags = [tag for _, tag in brown.tagged_words(categories='news')]
tag_dist = FreqDist(news_tags)
print("\nTop POS tags in news:")
print(tag_dist.most_common(10))

print("\n=== WordNet (lexical database) ===")
from nltk.corpus import wordnet

# Explore synonyms, definitions, examples for a word
synsets = wordnet.synsets('bank')
print(f"'bank' has {len(synsets)} senses:")
for s in synsets[:4]:
    print(f"  {s.name()}: {s.definition()}")
    if s.examples():
        print(f"    Example: '{s.examples()[0]}'")

## 11. Mini Project — News Article Analyzer

### The Task

Build a pipeline that analyzes a collection of news articles and produces:
1. Key phrase extraction (most frequent meaningful words)
2. Named entity counts
3. Sentiment scores
4. Topic summary statistics

In [ ]:
# ==================================================
# Simulate news articles (realistic content)
# ==================================================

articles = [
    {
        'title': 'Apple Announces Record Revenue',
        'text': """Apple Inc. reported record quarterly revenue of $120 billion yesterday,
        beating analyst expectations. CEO Tim Cook announced the results in Cupertino,
        California. The iPhone 15 sales drove most of the growth. Apple stock rose 5%
        on Wall Street. Investors are very excited about the company's AI initiatives.
        The company plans to hire 5,000 engineers in Austin, Texas next year."""
    },
    {
        'title': 'Climate Summit Fails to Reach Agreement',
        'text': """World leaders at the UN Climate Summit in Paris failed to reach a binding
        agreement on carbon emissions yesterday. The United States and China, the two largest
        polluters, could not agree on targets. Secretary General António Guterres expressed
        deep disappointment. Environmental groups are outraged. This is a terrible outcome
        for future generations."""
    },
    {
        'title': 'New Cancer Treatment Shows Promise',
        'text': """Scientists at Johns Hopkins University published amazing results showing
        a new immunotherapy treatment cured 80% of patients with advanced lung cancer.
        The study, published in Nature Medicine, involved 500 patients across 20 countries.
        Dr. Sarah Johnson led the research team. The treatment will be available at
        Memorial Sloan Kettering Cancer Center by 2025. Patients and doctors are
        incredibly hopeful."""
    },
]

print(f"Analyzing {len(articles)} news articles...")

In [ ]:
# ==================================================
# Full analysis pipeline
# ==================================================

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def analyze_article(article):
    text = article['text']
    tokens = word_tokenize(text)

    # 1. Key phrases (top non-stopword nouns and adjectives)
    pos_tags = pos_tag(tokens)
    meaningful = [
        lemmatizer.lemmatize(w.lower())
        for w, t in pos_tags
        if w.isalpha() and w.lower() not in stop_words
        and t in ('NN', 'NNS', 'NNP', 'JJ')
    ]
    key_phrases = FreqDist(meaningful).most_common(5)

    # 2. Named entities
    chunks = ne_chunk(pos_tags, binary=False)
    entities = {}
    for subtree in chunks:
        if isinstance(subtree, Tree):
            etype = subtree.label()
            etext = ' '.join(w for w, _ in subtree.leaves())
            entities.setdefault(etype, []).append(etext)

    # 3. Sentiment
    sentiment = sia.polarity_scores(text)

    return {
        'title': article['title'],
        'word_count': len([w for w in tokens if w.isalpha()]),
        'key_phrases': key_phrases,
        'entities': entities,
        'sentiment': sentiment
    }

results = [analyze_article(a) for a in articles]

# Display results
for r in results:
    print(f"\n{'='*60}")
    print(f"📰 {r['title']}")
    print(f"   Word count: {r['word_count']}")

    compound = r['sentiment']['compound']
    label = '🟢 Positive' if compound >= 0.05 else '🔴 Negative' if compound <= -0.05 else '⚪ Neutral'
    print(f"   Sentiment: {label} (compound={compound:.3f})")

    print(f"   Key terms: {[p for p, _ in r['key_phrases']]}")

    if r['entities']:
        print("   Entities:")
        for etype, names in r['entities'].items():
            unique_names = list(dict.fromkeys(names))[:3]  # deduplicate, take top 3
            print(f"     {etype}: {unique_names}")

In [ ]:
# Visualize sentiment comparison
fig, ax = plt.subplots(figsize=(9, 4))

titles    = [r['title'][:30] + '...' for r in results]
compounds = [r['sentiment']['compound'] for r in results]
pos       = [r['sentiment']['pos'] for r in results]
neg       = [r['sentiment']['neg'] for r in results]
neu       = [r['sentiment']['neu'] for r in results]

x = np.arange(len(results))
width = 0.25

ax.bar(x - width, pos, width, label='Positive', color='green', alpha=0.7)
ax.bar(x,         neu, width, label='Neutral',  color='gray',  alpha=0.7)
ax.bar(x + width, neg, width, label='Negative', color='red',   alpha=0.7)
ax.axhline(0, color='black', lw=0.5)

ax.set_xticks(x); ax.set_xticklabels(titles, fontsize=8)
ax.set_ylabel('VADER Score'); ax.set_title('News Article Sentiment Analysis')
ax.legend()
plt.tight_layout()
plt.show()

## 12. Common Pitfalls

### Pitfall 1: Not Downloading NLTK Data
```python
# Causes: LookupError: Resource 'punkt' not found.
nltk.download('punkt')  # run this ONCE before using tokenizers
nltk.download('all')    # downloads everything (300MB) — for offline use
```

### Pitfall 2: Order of Preprocessing Steps
Always: `lowercase → tokenize → remove stopwords → stem/lemmatize`. Stemming/lemmatizing before lowercasing may produce wrong results.

### Pitfall 3: Stemming for Production NLP
Stemming produces non-words (`"studi"`, `"comput"`). For production, always use **lemmatization** — it produces real words that are interpretable and give better downstream performance.

### Pitfall 4: VADER on Formal Text
VADER was designed for social media. For formal text (legal documents, scientific papers), use a trained classifier or a fine-tuned transformer model instead.

### Pitfall 5: NLTK for Production
NLTK is slow and its POS/NER accuracy is lower than spaCy or transformers. For production NLP, use **spaCy** (fast, accurate) or **HuggingFace** (state-of-the-art).

## 13. Interview Q&A

---

**Q1: What is tokenization and why is it the first step in NLP?**

> Tokenization splits raw text into discrete units (tokens) — usually words or subwords. It's the first step because all subsequent NLP operations (POS tagging, NER, embedding lookup) work on individual tokens, not raw strings. Tokenization handles edge cases like contractions (`"don't"` → `["do", "n't"]`), abbreviations (`"Dr."` — not a sentence end), punctuation (keep or remove?), and special characters. Different tokenization strategies exist: word-level, character-level, subword (BPE — used by GPT and BERT).

---

**Q2: What is the difference between stemming and lemmatization?**

> **Stemming** chops word suffixes using rules: `"running"` → `"run"`, `"studying"` → `"studi"`. Fast but produces non-words. **Lemmatization** looks up the base form in a dictionary (WordNet): `"running"` → `"run"`, `"better"` → `"good"`. Always produces real words but slower and requires POS context for best results. Use stemming for fast search indexing; use lemmatization for quality NLP applications where word meaning matters.

---

**Q3: What is TF-IDF and how does it differ from raw frequency?**

> **TF** (Term Frequency): how often a word appears in a document. The problem: common words like "the" always have high TF but carry no meaning. **IDF** (Inverse Document Frequency): `log(N / df)` where N is total documents and df is how many documents contain the word. Words that appear in every document get low IDF ("the" → near-zero IDF). **TF-IDF** = TF × IDF: words that appear a lot in ONE document but rarely across all documents get high scores — these are the distinctive, meaningful terms for that document.

---

**Q4: What is the bag-of-words model? What are its limitations?**

> The bag-of-words model represents a document as the frequency of each word, ignoring word order and grammar. `"The cat sat on the mat"` and `"The mat sat on the cat"` produce identical representations. Limitations: (1) loses all word order and context, (2) no semantic understanding — `"happy"` and `"joyful"` are completely separate features, (3) sparse and high-dimensional (vocabulary-sized vectors), (4) can't handle negation — `"not good"` and `"good"` look similar.

---

**Q5: NLTK vs spaCy — when would you use each?**

> **Use NLTK** when: learning NLP concepts (excellent for education), accessing built-in linguistic corpora (Gutenberg, Brown, WordNet), or doing research that requires classical NLP tools (grammar parsing, WordNet integration). **Use spaCy** when: building production NLP pipelines, needing fast processing (spaCy is 10-20x faster than NLTK), needing higher accuracy POS/NER, processing millions of documents, or building information extraction systems.

## 14. Resources

### Official
- **NLTK Documentation:** https://www.nltk.org/
- **NLTK Book (free, comprehensive):** https://www.nltk.org/book/
- **GitHub:** https://github.com/nltk/nltk

### Papers
- **VADER Sentiment Analysis:** https://ojs.aaai.org/index.php/ICWSM/article/view/14550

### Videos
- **NLP with Python and NLTK (Sentdex):** https://www.youtube.com/watch?v=FLZvOKSCkxY
- **Natural Language Processing Tutorial:** https://www.youtube.com/watch?v=X2vAabgKiuM

## 15. Summary & What's Next

### What You Learned

| Concept | Key Takeaway |
|---|---|
| **Tokenization** | `word_tokenize` / `sent_tokenize`; handles contractions, abbreviations, punctuation |
| **Stopwords** | Remove high-frequency, low-meaning words; NLTK has 23-language lists |
| **Stemming** | Rule-based suffix removal; fast but may produce non-words; use Porter/Snowball |
| **Lemmatization** | Dictionary-based root finding; always real words; use with POS tags for accuracy |
| **POS tagging** | Labels each word's grammatical role; Penn Treebank tagset |
| **NER** | Identifies people, organizations, places; NLTK's is basic — use spaCy for production |
| **FreqDist** | Word frequency counting + bigram collocations (PMI scoring) |
| **VADER** | Rule-based sentiment; great for social media; gives compound score |
| **Naive Bayes** | Bag-of-words text classifier; fast and interpretable; `show_most_informative_features()` |

### What's Next

**spaCy** — the modern, production-grade NLP library:
- Pre-trained pipelines (tokenizer → tagger → NER → dependency parser in one call)
- Much faster than NLTK (written in Cython)
- Rule-based matching (`Matcher`, `PhraseMatcher`)
- Integration with transformers via `spacy-transformers`